In [1]:
import os, time, yaml
from pyspark.sql import SparkSession, Window, functions as F

with open("/home/jovyan/work/config.yaml") as f:
    cfg = yaml.safe_load(f)

HDFS = cfg["hdfs_base"]                              # hdfs://namenode:9000
RAW = HDFS + cfg["paths"]["raw"]                     # zona raw: el CSV tal cual
CURATED = HDFS + cfg["paths"]["curated"]             # zona curada: Parquet limpio
LANG_INFO = HDFS + cfg["paths"]["language_info"]     # tabla auxiliar de idiomas
RESULTS = HDFS + cfg["paths"]["results"]             # zona de resultados

# Igual que en el Tema 5: actuamos como root en HDFS, ANTES de crear la sesión
os.environ["HADOOP_USER_NAME"] = "root"

spark = (SparkSession.builder
         .appName("wikirank-experiments")
         .master(cfg["spark"]["master"])                             # en clase: local[*]
         .config("spark.driver.host", cfg["spark"]["driver_host"])
         .config("spark.executor.memory", cfg["spark"]["executor_memory"])
         .config("spark.hadoop.dfs.replication", cfg["spark"]["dfs_replication"])
         .getOrCreate())
spark


In [ ]:
articles = spark.read.parquet(CURATED)
lang_info = spark.read.parquet(LANG_INFO)
RAW_FILE = RAW + "/" + os.path.basename(cfg["input"]["local_csv"])
schema = ("Wikipedia_article_URL STRING, Language STRING, Page_ID LONG, "
          "Title STRING, WikiRank_score DOUBLE, WikiRank_URL STRING")
raw = spark.read.schema(schema).option("header", True).option("escape", '"').csv(RAW_FILE)

In [ ]:
import csv

lines = spark.sparkContext.textFile(RAW_FILE)
header = lines.first()

def parse(line):
    # Como l.split(","), pero respetando las comas que hay dentro de algunos títulos
    f = next(csv.reader([line]))
    try:
        lang, score = f[1].strip().lower(), float(f[4])
    except (IndexError, ValueError):
        return []                                        # fila rota: se descarta
    return [(lang, score)] if lang and 0 <= score <= 100 else []

t = time.time()
result = (lines.filter(lambda l: l != header)            # quitar la cabecera
               .flatMap(parse)                           # MAP: (idioma, score)
               .mapValues(lambda s: (s, 1))              # (idioma, (score, 1))
               .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))   # SHUFFLE + REDUCE
               .mapValues(lambda x: (x[1], round(x[0] / x[1], 2)))    # (idioma, (n, media))
               .sortBy(lambda kv: -kv[1][0])
               .collect())
print(f"RDD: {time.time() - t:.1f} s")
for lang, (n, avg) in result[:10]:
    print(lang, n, avg)


RDD: 66.2 s
en 6860430 24.71
de 2930971 18.63
fr 2626210 16.12
sv 2591505 20.02
nl 2163715 12.73
ru 1991787 18.18
es 1905093 18.77
it 1875385 13.63
pl 1624175 17.03
zh 1433026 20.31


**Optimización**

In [4]:
t = time.time()
raw.agg(F.avg("WikiRank_score")).show()
print(f"CSV:     {time.time() - t:.1f} s")

t = time.time()
articles.agg(F.avg("WikiRank_score")).show()
print(f"Parquet: {time.time() - t:.1f} s")

# ¿Por qué? En la línea FileScan, ReadSchema: Parquet solo lee la columna WikiRank_score
articles.agg(F.avg("WikiRank_score")).explain()

+-------------------+
|avg(WikiRank_score)|
+-------------------+
| 17.932223616843302|
+-------------------+

CSV:     30.7 s
+-------------------+
|avg(WikiRank_score)|
+-------------------+
| 17.932223616844187|
+-------------------+

Parquet: 1.5 s
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[avg(WikiRank_score#2)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=92]
      +- HashAggregate(keys=[], functions=[partial_avg(WikiRank_score#2)])
         +- Project [WikiRank_score#2]
            +- FileScan parquet [WikiRank_score#2,Language#4] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://namenode:9000/curated/articles], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<WikiRank_score:double>




In [5]:
# La consulta de P3b: une los 47 M de artículos con la escritura de su lengua y calcula la mediana
def q3_expr(info_table):
    return (articles.join(info_table, "Language", "left")
                    .groupBy("script")
                    .agg(F.count("*").alias("n_articles"),
                         F.percentile_approx("WikiRank_score", 0.5, 10000).alias("median_score")))

# ANTES: sin broadcast (-1 = nunca). Spark hace sort-merge join: shuffle de las DOS tablas
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
lang_sub = lang_info.select("Language", "script")

q_smj = q3_expr(lang_sub)
q_smj.explain()                      # busca SortMergeJoin y los dos Exchange

t = time.time()
q_smj.write.format("noop").mode("overwrite").save()   # noop: calcula todo sin guardar
print(f"Sort-merge join (sin broadcast): {time.time() - t:.1f} s")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- ObjectHashAggregate(keys=[script#8], functions=[count(1), percentile_approx(WikiRank_score#2, 0.5, 10000, 0, 0)])
   +- Exchange hashpartitioning(script#8, 200), ENSURE_REQUIREMENTS, [plan_id=134]
      +- ObjectHashAggregate(keys=[script#8], functions=[partial_count(1), partial_percentile_approx(WikiRank_score#2, 0.5, 10000, 0, 0)])
         +- Project [WikiRank_score#2, script#8]
            +- SortMergeJoin [Language#4], [Language#5], LeftOuter
               :- Sort [Language#4 ASC NULLS FIRST], false, 0
               :  +- Exchange hashpartitioning(Language#4, 200), ENSURE_REQUIREMENTS, [plan_id=126]
               :     +- FileScan parquet [WikiRank_score#2,Language#4] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://namenode:9000/curated/articles], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<WikiRank_score:double>
               +- Sort [Language#5 ASC NULLS FI

In [6]:
# Restauramos el umbral por defecto (10 MB)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

# Forzamos explícitamente el broadcast de la tabla auxiliar pequeña
q_bc = q3_expr(F.broadcast(lang_sub))
q_bc.explain()  # Muestra el plan de ejecución: verás "BroadcastExchange" y "BroadcastHashJoin"

t = time.time()
q_bc.write.format("noop").mode("overwrite").save()
print(f"Broadcast join: {time.time() - t:.1f} s")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- ObjectHashAggregate(keys=[script#8], functions=[count(1), percentile_approx(WikiRank_score#2, 0.5, 10000, 0, 0)])
   +- Exchange hashpartitioning(script#8, 200), ENSURE_REQUIREMENTS, [plan_id=380]
      +- ObjectHashAggregate(keys=[script#8], functions=[partial_count(1), partial_percentile_approx(WikiRank_score#2, 0.5, 10000, 0, 0)])
         +- Project [WikiRank_score#2, script#8]
            +- BroadcastHashJoin [Language#4], [Language#5], LeftOuter, BuildRight, false, false
               :- FileScan parquet [WikiRank_score#2,Language#4] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://namenode:9000/curated/articles], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<WikiRank_score:double>
               +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=375]
                  +- Filter isnotnull(Language#5)
                   

In [7]:
# Desafío F.3: data skew. ¿Cuánto más grande es la lengua mayor que la mediana?
counts = articles.groupBy("Language").count().orderBy(F.desc("count"))
counts.show(5)
median = counts.approxQuantile("count", [0.5], 0.0)[0]
top = counts.first()["count"]
print(f"Mayor: {top} | mediana: {median:.0f} | {top / median:.1f} veces la mediana")

+--------+-------+
|Language|  count|
+--------+-------+
|      en|6860430|
|      de|2930971|
|      fr|2626210|
|      sv|2591505|
|      nl|2163715|
+--------+-------+
only showing top 5 rows
Mayor: 6860430 | mediana: 433220 | 15.8 veces la mediana


In [8]:
# Join con shuffle por lengua: la partición del inglés es mucho mayor que el resto (skew)
avg_by_lang = articles.groupBy("Language").agg(F.avg("WikiRank_score").alias("avg_score"))
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)       # forzar join con shuffle

# 1) SIN AQE: una sola tarea procesa todo el inglés
spark.conf.set("spark.sql.adaptive.enabled", False)
t = time.time()
articles.join(avg_by_lang, "Language").write.format("noop").mode("overwrite").save()
print(f"Sin AQE: {time.time() - t:.1f} s")

# 2) CON AQE + skewJoin: Spark parte la partición del inglés en trozos
spark.conf.set("spark.sql.adaptive.enabled", True)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", True)
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", 3)                  # >3x la mediana = skew
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "32m")    # y >32 MB
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "16m")                # trozos de ~16 MB
t = time.time()
articles.join(avg_by_lang, "Language").write.format("noop").mode("overwrite").save()
print(f"Con AQE: {time.time() - t:.1f} s")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)   # restaurar el valor por defecto

Sin AQE: 31.8 s
Con AQE: 32.3 s


In [9]:
#spark.stop()